# 🇬🇧 UK Retail Performance Hub
## Phase 2 — Analytical Models

**What this notebook builds:**

| # | Model | What it answers |
|---|---|---|
| 1 | RFM Segmentation | Who are our best customers? |
| 2 | Cohort Retention | Do customers come back after first purchase? |
| 3 | Churn Scoring | Which customers are at risk of leaving? |
| 4 | Revenue Forecasting | What does next month's revenue look like? |

Each model exports a CSV to `data/powerbi/` so Power BI can read it directly.

**Prerequisites:** All 4 Phase 1 notebooks must have been run first.

---

### Cell 1 — Mount Drive & set up paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

ROOT          = Path('/content/drive/MyDrive/uk-retail-performance-hub')
PROCESSED_DIR = ROOT / 'data' / 'processed'
POWERBI_DIR   = ROOT / 'data' / 'powerbi'
POWERBI_DIR.mkdir(parents=True, exist_ok=True)

print('✓ Drive mounted')
print(f'  Reading from : {PROCESSED_DIR}')
print(f'  Exporting to : {POWERBI_DIR}')

### Cell 2 — Install libraries & imports

In [ ]:
!pip install scikit-learn statsmodels --quiet

import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Chart style
plt.rcParams['figure.figsize']  = (12, 5)
plt.rcParams['axes.spines.top']    = False
plt.rcParams['axes.spines.right']  = False
plt.rcParams['font.family']        = 'sans-serif'
sns.set_palette('muted')

pd.set_option('display.float_format', '{:,.2f}'.format)
print('✓ All libraries ready')

### Cell 3 — Load clean data

In [ ]:
df = pd.read_parquet(PROCESSED_DIR / 'uci_clean.parquet')

# Ensure correct dtypes
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Revenue']     = pd.to_numeric(df['Revenue'], errors='coerce')

# Reference date — the day after the last transaction in the dataset
REFERENCE_DATE = df['InvoiceDate'].max() + pd.Timedelta(days=1)

print(f'✓ Data loaded: {len(df):,} rows')
print(f'  Customers    : {df["CustomerID"].nunique():,}')
print(f'  Date range   : {df["InvoiceDate"].min().date()} → {df["InvoiceDate"].max().date()}')
print(f'  Reference date for RFM/churn: {REFERENCE_DATE.date()}')

---
## Model 1 — RFM Customer Segmentation

**What is RFM?**
RFM scores each customer on three dimensions:
- **Recency** — how recently did they buy? (lower days = better)
- **Frequency** — how many orders have they placed?
- **Monetary** — how much have they spent in total?

Each dimension is scored 1–5, then combined to segment customers into groups like Champions, Loyal, At Risk and Lost.

---

### Cell 4 — Calculate RFM scores

In [ ]:
# Build one row per customer with R, F, M values
rfm = df.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate',  lambda x: (REFERENCE_DATE - x.max()).days),
    Frequency = ('InvoiceNo',    'nunique'),
    Monetary  = ('Revenue',      'sum')
).reset_index()

rfm['Monetary'] = rfm['Monetary'].round(2)

# Score each dimension 1–5 using quintiles
# Recency: lower days = better = higher score (hence reversed labels)
rfm['R_Score'] = pd.qcut(rfm['Recency'],   q=5, labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'),  q=5, labels=[1,2,3,4,5]).astype(int)

# Combined RFM score (string, e.g. '555' = best customer)
rfm['RFM_Score']  = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm['RFM_Total']  = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

print(f'✓ RFM scores calculated for {len(rfm):,} customers')
print(f'\nRFM score distribution:')
print(rfm[['R_Score','F_Score','M_Score','RFM_Total']].describe().round(1).to_string())

### Cell 5 — Assign customer segments

In [ ]:
def assign_segment(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'Recent Customers'
    elif r >= 3 and f <= 2 and m >= 3:
        return 'Potential Loyalists'
    elif r == 3 and f == 3:
        return 'Needs Attention'
    elif r <= 2 and f >= 3 and m >= 3:
        return 'At Risk'
    elif r <= 2 and f >= 4:
        return 'Cant Lose Them'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Lost'
    else:
        return 'Hibernating'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)

# Segment summary
seg_summary = rfm.groupby('Segment').agg(
    Customers     = ('CustomerID', 'count'),
    Avg_Recency   = ('Recency',   'mean'),
    Avg_Frequency = ('Frequency', 'mean'),
    Avg_Monetary  = ('Monetary',  'mean'),
    Total_Revenue = ('Monetary',  'sum')
).round(1).sort_values('Total_Revenue', ascending=False)

seg_summary['Revenue_Share_%'] = (seg_summary['Total_Revenue'] / seg_summary['Total_Revenue'].sum() * 100).round(1)

print('RFM SEGMENT SUMMARY')
print('=' * 75)
print(seg_summary.to_string())

### Cell 6 — Visualise RFM segments

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('RFM Customer Segmentation', fontsize=14, fontweight='bold', y=1.02)

# Chart 1: Customer count by segment
seg_count = rfm['Segment'].value_counts()
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(seg_count)))
axes[0].barh(seg_count.index, seg_count.values, color=colors)
axes[0].set_title('Customers per Segment')
axes[0].set_xlabel('Number of Customers')
for i, v in enumerate(seg_count.values):
    axes[0].text(v + 5, i, str(v), va='center', fontsize=9)

# Chart 2: Revenue share by segment
seg_rev = rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=True)
axes[1].barh(seg_rev.index, seg_rev.values / 1000, color=plt.cm.Blues(np.linspace(0.3, 0.9, len(seg_rev))))
axes[1].set_title('Revenue by Segment (£000s)')
axes[1].set_xlabel('Total Revenue (£000s)')
for i, v in enumerate(seg_rev.values):
    axes[1].text(v/1000 + 5, i, f'£{v/1000:,.0f}k', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(POWERBI_DIR / 'rfm_segments.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Chart saved')

### Cell 7 — Export RFM data for Power BI

In [ ]:
rfm.to_csv(POWERBI_DIR / 'rfm_scores.csv', index=False)
seg_summary.reset_index().to_csv(POWERBI_DIR / 'rfm_segment_summary.csv', index=False)

print('✓ RFM exports saved:')
print(f'  rfm_scores.csv          — {len(rfm):,} rows (one per customer)')
print(f'  rfm_segment_summary.csv — {len(seg_summary)} rows (one per segment)')

# Key insight
champions    = seg_summary.loc['Champions']
at_risk      = seg_summary.loc['At Risk'] if 'At Risk' in seg_summary.index else None
print(f'\n💡 Key insight:')
print(f'   Champions represent {champions["Customers"]:.0f} customers ({champions["Revenue_Share_%"]}% of revenue)')
if at_risk is not None:
    print(f'   At Risk customers: {at_risk["Customers"]:.0f} customers with £{at_risk["Total_Revenue"]:,.0f} revenue at stake')

---
## Model 2 — Cohort Retention Analysis

**What is cohort retention?**
Groups customers by the month they first purchased, then tracks what percentage return in each subsequent month.
A healthy business retains customers over time — a high drop-off after month 0 signals a re-engagement opportunity.

---

### Cell 8 — Build cohort retention matrix

In [ ]:
# Get each customer's cohort month and activity month
cohort_df = df.groupby(['CustomerID', 'YearMonth', 'CohortMonth']).size().reset_index(name='Transactions')

# Calculate month offset (0 = acquisition month)
def month_diff(activity, cohort):
    ay, am = int(activity[:4]), int(activity[5:])
    cy, cm = int(cohort[:4]),   int(cohort[5:])
    return (ay * 12 + am) - (cy * 12 + cm)

cohort_df['MonthOffset'] = cohort_df.apply(
    lambda r: month_diff(r['YearMonth'], r['CohortMonth']), axis=1
)

# Count unique customers per cohort per offset
cohort_counts = cohort_df.groupby(['CohortMonth', 'MonthOffset'])['CustomerID'].nunique().reset_index()
cohort_pivot  = cohort_counts.pivot(index='CohortMonth', columns='MonthOffset', values='CustomerID')

# Convert to retention % (divide every column by month 0 = cohort size)
cohort_size      = cohort_pivot[0]
retention_matrix = cohort_pivot.divide(cohort_size, axis=0).round(3) * 100

print(f'✓ Cohort matrix built: {len(retention_matrix)} cohorts × {len(retention_matrix.columns)} months')
print(f'\nRetention % (first 6 months shown):')
print(retention_matrix.iloc[:, :6].round(1).to_string())

### Cell 9 — Visualise retention heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

sns.heatmap(
    retention_matrix.iloc[:, :12],
    annot=True,
    fmt='.0f',
    cmap='YlOrRd_r',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Retention %'},
    vmin=0,
    vmax=100
)

ax.set_title('Customer Cohort Retention Heatmap\n(% of cohort still active by month)', 
             fontsize=13, fontweight='bold')
ax.set_xlabel('Months since first purchase')
ax.set_ylabel('Cohort (acquisition month)')

plt.tight_layout()
plt.savefig(POWERBI_DIR / 'cohort_retention_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Heatmap saved')

### Cell 10 — Retention insight summary

In [ ]:
# Average retention by month offset across all cohorts
avg_retention = retention_matrix.mean().round(1)

print('AVERAGE RETENTION BY MONTH (across all cohorts)')
print('=' * 45)
for month, pct in avg_retention.items():
    bar = '█' * int(pct / 5)
    print(f'  Month {month:>2} : {pct:>5.1f}%  {bar}')

month1_retention = avg_retention.get(1, 0)
month3_retention = avg_retention.get(3, 0)

print(f'\n💡 Key insight:')
print(f'   Only {month1_retention:.0f}% of customers return after their first purchase')
print(f'   By month 3, retention drops to {month3_retention:.0f}%')
print(f'   → This points to a strong re-engagement opportunity in the 30–90 day window')

# Export
retention_matrix.reset_index().to_csv(POWERBI_DIR / 'cohort_retention.csv', index=False)
avg_retention.reset_index().rename(columns={0:'AvgRetention_Pct'}).to_csv(
    POWERBI_DIR / 'avg_retention_by_month.csv', index=False
)
print('\n✓ Cohort data exported to Power BI folder')

---
## Model 3 — Churn Scoring

**What is churn scoring?**
Uses customer behaviour features to predict the probability that a customer will **not** return.
We define a churned customer as one who bought before the last 90 days of the dataset but has not purchased since.
A logistic regression model scores every customer with a churn probability (0–100%).

---

### Cell 11 — Build churn features

In [ ]:
# Churn window: 90 days before dataset end
churn_cutoff = REFERENCE_DATE - pd.Timedelta(days=90)

# Features: use activity up to the churn cutoff
train_data = df[df['InvoiceDate'] < churn_cutoff]

features = train_data.groupby('CustomerID').agg(
    Recency         = ('InvoiceDate',  lambda x: (churn_cutoff - x.max()).days),
    Frequency       = ('InvoiceNo',    'nunique'),
    Monetary        = ('Revenue',      'sum'),
    AvgOrderValue   = ('Revenue',      'mean'),
    UniqueProducts  = ('StockCode',    'nunique'),
    TotalItems      = ('Quantity',     'sum'),
    IsUK            = ('IsUK',         'first')
).reset_index()

# Label: did the customer purchase in the final 90 days?
active_in_window = df[df['InvoiceDate'] >= churn_cutoff]['CustomerID'].unique()
features['Churned'] = (~features['CustomerID'].isin(active_in_window)).astype(int)

churn_rate = features['Churned'].mean() * 100
print(f'✓ Churn features built for {len(features):,} customers')
print(f'  Churn cutoff date: {churn_cutoff.date()}')
print(f'  Observed churn rate: {churn_rate:.1f}%')
print(f'  Churned : {features["Churned"].sum():,}')
print(f'  Retained: {(features["Churned"]==0).sum():,}')

### Cell 12 — Train churn model

In [ ]:
feature_cols = ['Recency','Frequency','Monetary','AvgOrderValue','UniqueProducts','TotalItems','IsUK']

X = features[feature_cols].fillna(0)
y = features['Churned']

# Scale features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_train, y_train)

print('MODEL PERFORMANCE ON TEST SET')
print('=' * 45)
print(classification_report(y_test, model.predict(X_test), 
                             target_names=['Retained','Churned']))

# Feature importance
coef_df = pd.DataFrame({
    'Feature'     : feature_cols,
    'Coefficient' : model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

print('Feature importance (by coefficient magnitude):')
print(coef_df.to_string(index=False))

### Cell 13 — Score all customers & export

In [ ]:
# Score every customer with churn probability
X_all = scaler.transform(features[feature_cols].fillna(0))
features['ChurnProbability'] = (model.predict_proba(X_all)[:, 1] * 100).round(1)

# Risk tier
def risk_tier(prob):
    if prob >= 75:  return 'High Risk'
    elif prob >= 50: return 'Medium Risk'
    elif prob >= 25: return 'Low Risk'
    else:           return 'Safe'

features['ChurnRiskTier'] = features['ChurnProbability'].apply(risk_tier)

# Add RFM segment to churn table
churn_output = features.merge(rfm[['CustomerID','Segment','RFM_Total']], on='CustomerID', how='left')

# Summary by risk tier
tier_summary = churn_output.groupby('ChurnRiskTier').agg(
    Customers        = ('CustomerID',       'count'),
    Avg_Churn_Prob   = ('ChurnProbability', 'mean'),
    Total_Monetary   = ('Monetary',         'sum')
).round(1)

tier_summary['Revenue_At_Risk'] = tier_summary.apply(
    lambda r: r['Total_Monetary'] if 'Risk' in r.name else 0, axis=1
)

print('CHURN RISK TIER SUMMARY')
print('=' * 55)
print(tier_summary.sort_values('Avg_Churn_Prob', ascending=False).to_string())

high_risk_rev = tier_summary.loc['High Risk', 'Total_Monetary'] if 'High Risk' in tier_summary.index else 0
print(f'\n💡 Key insight:')
print(f'   High risk customers represent £{high_risk_rev:,.0f} in historical revenue')
print(f'   → Re-engagement campaigns targeting this group offer the highest ROI')

churn_output.to_csv(POWERBI_DIR / 'churn_scores.csv', index=False)
tier_summary.reset_index().to_csv(POWERBI_DIR / 'churn_tier_summary.csv', index=False)
print('\n✓ Churn scores exported to Power BI folder')

### Cell 14 — Visualise churn risk distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Customer Churn Risk Analysis', fontsize=14, fontweight='bold')

# Chart 1: Distribution of churn probabilities
axes[0].hist(features['ChurnProbability'], bins=20, color='#E74C3C', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution of Churn Probability Scores')
axes[0].set_xlabel('Churn Probability (%)')
axes[0].set_ylabel('Number of Customers')
axes[0].axvline(x=50, color='black', linestyle='--', linewidth=1, label='50% threshold')
axes[0].legend()

# Chart 2: Revenue by risk tier
tier_rev = churn_output.groupby('ChurnRiskTier')['Monetary'].sum().sort_values(ascending=False)
tier_colors = {'High Risk':'#E74C3C','Medium Risk':'#E67E22','Low Risk':'#F1C40F','Safe':'#2ECC71'}
colors = [tier_colors.get(t, '#95A5A6') for t in tier_rev.index]
axes[1].bar(tier_rev.index, tier_rev.values / 1000, color=colors, edgecolor='white')
axes[1].set_title('Historical Revenue by Churn Risk Tier')
axes[1].set_xlabel('Risk Tier')
axes[1].set_ylabel('Revenue (£000s)')
for i, v in enumerate(tier_rev.values):
    axes[1].text(i, v/1000 + 5, f'£{v/1000:,.0f}k', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(POWERBI_DIR / 'churn_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Chart saved')

---
## Model 4 — Revenue Forecasting

**What does this do?**
Uses Holt-Winters Exponential Smoothing — a method that accounts for both trend and seasonality — to forecast the next 3 months of revenue.
This is the kind of forward-looking analysis that directly supports commercial planning and budgeting.

---

### Cell 15 — Prepare monthly revenue time series

In [ ]:
monthly = (
    df.groupby('YearMonth')['Revenue']
    .sum()
    .reset_index()
    .rename(columns={'Revenue': 'TotalRevenue'})
)

monthly['Date'] = pd.to_datetime(monthly['YearMonth'] + '-01')
monthly = monthly.sort_values('Date').set_index('Date')

# Remove the last month (Dec 2011) as it's partial (only 9 days)
monthly = monthly.iloc[:-1]

print('Monthly revenue series (used for forecasting):')
print(monthly[['YearMonth','TotalRevenue']].to_string())
print(f'\n  Months in series : {len(monthly)}')
print(f'  Mean monthly rev : £{monthly["TotalRevenue"].mean():,.0f}')
print(f'  Peak month       : {monthly["TotalRevenue"].idxmax().strftime("%b %Y")} (£{monthly["TotalRevenue"].max():,.0f})')

### Cell 16 — Fit forecasting model & predict 3 months ahead

In [ ]:
# Holt-Winters with additive trend
# With only 13 months we can't fit a full seasonal model,
# so we use trend-only which is honest given the data length
hw_model = ExponentialSmoothing(
    monthly['TotalRevenue'],
    trend='add',
    damped_trend=True
).fit(optimized=True)

# Forecast 3 months ahead
forecast_steps  = 3
forecast_values = hw_model.forecast(forecast_steps)

# Confidence interval (±1.5 standard deviations of residuals)
residuals  = hw_model.resid
resid_std  = residuals.std()
margin     = resid_std * 1.5

forecast_df = pd.DataFrame({
    'Date'         : pd.date_range(monthly.index[-1] + pd.DateOffset(months=1), periods=forecast_steps, freq='MS'),
    'Forecast'     : forecast_values.values.round(2),
    'Lower_Bound'  : (forecast_values.values - margin).round(2),
    'Upper_Bound'  : (forecast_values.values + margin).round(2),
    'Type'         : 'Forecast'
})

print('3-MONTH REVENUE FORECAST')
print('=' * 55)
for _, row in forecast_df.iterrows():
    print(f"  {row['Date'].strftime('%b %Y')}:  £{row['Forecast']:>10,.0f}  "
          f"(range: £{row['Lower_Bound']:,.0f} – £{row['Upper_Bound']:,.0f})")

fitted_df = pd.DataFrame({
    'Date'     : monthly.index,
    'Actual'   : monthly['TotalRevenue'].values,
    'Fitted'   : hw_model.fittedvalues.values.round(2),
    'Type'     : 'Actual'
})

### Cell 17 — Visualise forecast

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

# Actual revenue
ax.plot(fitted_df['Date'], fitted_df['Actual'] / 1000,
        marker='o', linewidth=2, color='#2C3E50', label='Actual Revenue', zorder=3)

# Fitted values
ax.plot(fitted_df['Date'], fitted_df['Fitted'] / 1000,
        linestyle='--', linewidth=1.5, color='#7F8C8D', label='Model Fit', alpha=0.7)

# Forecast
ax.plot(forecast_df['Date'], forecast_df['Forecast'] / 1000,
        marker='o', linewidth=2.5, color='#E74C3C', linestyle='--', label='Forecast', zorder=3)

# Confidence band
ax.fill_between(
    forecast_df['Date'],
    forecast_df['Lower_Bound'] / 1000,
    forecast_df['Upper_Bound'] / 1000,
    alpha=0.2, color='#E74C3C', label='Confidence Range'
)

# Divider line between actual and forecast
ax.axvline(x=monthly.index[-1], color='gray', linestyle=':', linewidth=1)
ax.text(monthly.index[-1], ax.get_ylim()[1] * 0.95, ' Forecast →',
        color='gray', fontsize=9)

ax.set_title('Monthly Revenue: Actual vs Forecast', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (£000s)')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'£{x:,.0f}k'))
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(POWERBI_DIR / 'revenue_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Forecast chart saved')

### Cell 18 — Export forecast data for Power BI

In [ ]:
# Combine actual + forecast into one table for Power BI
actual_export = fitted_df[['Date','Actual','Fitted']].copy()
actual_export.columns = ['Date','Revenue','Fitted']
actual_export['IsForcast'] = 0
actual_export['Lower_Bound'] = None
actual_export['Upper_Bound'] = None

forecast_export = forecast_df[['Date','Forecast','Lower_Bound','Upper_Bound']].copy()
forecast_export.columns = ['Date','Revenue','Lower_Bound','Upper_Bound']
forecast_export['IsForcast'] = 1
forecast_export['Fitted'] = None

combined = pd.concat([actual_export, forecast_export], ignore_index=True)
combined['Date'] = combined['Date'].astype(str)
combined.to_csv(POWERBI_DIR / 'revenue_forecast.csv', index=False)

print('✓ Forecast exported')
print(combined[['Date','Revenue','IsForcast']].to_string(index=False))

---
## Phase 2 Complete — Summary

### Cell 19 — Full output summary

In [ ]:
print('=' * 60)
print('  PHASE 2 COMPLETE — FILES EXPORTED TO POWER BI FOLDER')
print('=' * 60)

exports = list(POWERBI_DIR.glob('*.csv')) + list(POWERBI_DIR.glob('*.png'))
exports.sort()

for f in exports:
    size_kb = f.stat().st_size / 1000
    print(f'  {f.name:<45} {size_kb:>7.1f} KB')

print('\n' + '=' * 60)
print('  KEY INSIGHTS SUMMARY')
print('=' * 60)

# RFM
top_seg = seg_summary.iloc[0]
print(f'\n  RFM:')
print(f'  Top segment by revenue: {top_seg.name} ({top_seg["Revenue_Share_%"]}% of total revenue)')

# Retention
m1 = avg_retention.get(1, 0)
print(f'\n  Cohort Retention:')
print(f'  Month-1 retention: {m1:.0f}% — {100-m1:.0f}% of customers do not return after first purchase')

# Churn
high_risk_count = (features['ChurnProbability'] >= 75).sum()
print(f'\n  Churn:')
print(f'  {high_risk_count:,} customers scored as High Risk (≥75% churn probability)')

# Forecast
next_month = forecast_df.iloc[0]
print(f'\n  Forecast:')
print(f'  {next_month["Date"].strftime("%b %Y")} forecast: £{next_month["Forecast"]:,.0f}')

print('\n' + '=' * 60)
print('  Next step: Build the Power BI dashboard using these CSVs')
print('=' * 60)